# Churn Prediction Model Comparison

In [ ]:
import os
# Works regardless of where the notebook is run from
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
MODELS_DIR   = os.path.join(BASE_DIR, 'models')
PROCESSED_DIR = os.path.join(BASE_DIR, 'processed')
VISUALS_DIR   = os.path.join(BASE_DIR, 'visuals')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(VISUALS_DIR, exist_ok=True)

In [ ]:
import joblib, os
import pandas as pd
import numpy as np

# Load test datasets
X_test = joblib.load(os.path.join(PROCESSED_DIR, 'X_test.pkl'))
y_test = joblib.load(os.path.join(PROCESSED_DIR, 'y_test.pkl'))

# Load results summary CSV from Phase 5
results_df = pd.read_csv(os.path.join(MODELS_DIR, 'results_summary.csv'))
print("Results summary table loaded:")
print(results_df)

### CHART 1 — Accuracy vs F1 Score Bar Chart
A grouped bar chart displaying the comparison between Accuracy and F1 Score for all 6 trained models.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Bar(
        name='Accuracy',
        x=results_df['Model'],
        y=results_df['Accuracy'],
        text=results_df['Accuracy'],
        textposition='auto',
        marker_color='rgb(31, 119, 180)'
    ),
    go.Bar(
        name='F1 Score',
        x=results_df['Model'],
        y=results_df['F1'],
        text=results_df['F1'],
        textposition='auto',
        marker_color='coral'
    )
])

fig.update_layout(
    barmode='group',
    title="Accuracy vs F1 Score — All Models",
    xaxis_title="Model",
    yaxis_title="Score",
    legend_title="Metric",
    template="plotly_white"
)

# Save chart as HTML
save_path_chart1 = os.path.join(VISUALS_DIR, 'accuracy_vs_f1.html')
fig.write_html(save_path_chart1)
print(f"Chart saved → {save_path_chart1}")
fig.show()

### CHART 2 — Multi-Model ROC Curves (all on one graph)
Plots the Receiver Operating Characteristic (ROC) curves and calculates the Area Under the Curve (AUC) for all 6 models to evaluate their capability to distinguish between classes.

In [ ]:
from sklearn.metrics import roc_curve, auc

fig = go.Figure()

# Add standard random-classifier reference line
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    line=dict(dash='dash', color='gray'),
    name='Random Classifier (AUC = 0.50)'
))

model_files = {
    'Logistic Regression': 'logistic_regression.pkl',
    'Decision Tree': 'decision_tree.pkl',
    'Random Forest': 'random_forest.pkl',
    'KNN': 'knn.pkl',
    'SVM': 'svm.pkl',
    'XGBoost': 'xgboost.pkl'
}

for model_name, filename in model_files.items():
    model_path = os.path.join(MODELS_DIR, filename)
    model = joblib.load(model_path)
    
    # Get probabilities
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.decision_function(X_test)
        
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f"{model_name} (AUC = {roc_auc:.2f})"
    ))

fig.update_layout(
    title="ROC Curves — All Models",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
    legend=dict(x=0.68, y=0.12)
)

# Save chart as HTML
save_path_chart2 = os.path.join(VISUALS_DIR, 'roc_curves.html')
fig.write_html(save_path_chart2)
print(f"Chart saved → {save_path_chart2}")
fig.show()

### CHART 3 — Confusion Matrix Grid (side by side)
A 2x3 grid displaying heatmaps of confusion matrices for all 6 models side-by-side to understand false positives and false negatives.

In [ ]:
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix

model_names = list(model_files.keys())
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=model_names,
    horizontal_spacing=0.1,
    vertical_spacing=0.18
)

for idx, (model_name, filename) in enumerate(model_files.items()):
    model_path = os.path.join(MODELS_DIR, filename)
    model = joblib.load(model_path)
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    row = (idx // 3) + 1
    col = (idx % 3) + 1
    
    heatmap = go.Heatmap(
        z=cm,
        x=['No Churn', 'Churn'],
        y=['No Churn', 'Churn'],
        colorscale='Blues',
        showscale=False,
        text=[[str(val) for val in row_vals] for row_vals in cm],
        texttemplate="%{text}",
        textfont={"size": 14}
    )
    
    fig.add_trace(heatmap, row=row, col=col)

fig.update_yaxes(autorange="reversed")

fig.update_layout(
    title_text="Confusion Matrices — All Models",
    height=600,
    width=900,
    template="plotly_white"
)

# Save chart as HTML
save_path_chart3 = os.path.join(VISUALS_DIR, 'confusion_matrices.html')
fig.write_html(save_path_chart3)
print(f"Chart saved → {save_path_chart3}")
fig.show()

### CHART 4 — Feature Importance Chart
A horizontal grouped bar chart showing the comparison between the top 15 features extracted from Random Forest and XGBoost.

In [ ]:
# Load models
rf_model = joblib.load(os.path.join(MODELS_DIR, 'random_forest.pkl'))
xgb_model = joblib.load(os.path.join(MODELS_DIR, 'xgboost.pkl'))

# Retrieve feature importances
features = X_test.columns.tolist()
rf_importances = rf_model.feature_importances_
xgb_importances = xgb_model.feature_importances_

importances_df = pd.DataFrame({
    'Feature': features,
    'Random Forest': rf_importances,
    'XGBoost': xgb_importances
})

# Select top 15 features based on average importance score
importances_df['Average'] = (importances_df['Random Forest'] + importances_df['XGBoost']) / 2
top_15_features = importances_df.sort_values(by='Average', ascending=False).head(15).copy()

# Reverse ordering for plotting bottom-up on horizontal chart
top_15_features = top_15_features.iloc[::-1]

fig = go.Figure(data=[
    go.Bar(
        name='Random Forest',
        y=top_15_features['Feature'],
        x=top_15_features['Random Forest'],
        orientation='h',
        marker_color='green'
    ),
    go.Bar(
        name='XGBoost',
        y=top_15_features['Feature'],
        x=top_15_features['XGBoost'],
        orientation='h',
        marker_color='#FFBF00' # Amber hex code
    )
])

fig.update_layout(
    title="Feature Importance — Random Forest vs XGBoost",
    barmode='group',
    xaxis_title="Importance Score",
    yaxis_title="Feature Name",
    template="plotly_white",
    height=700
)

# Save chart as HTML
save_path_chart4 = os.path.join(VISUALS_DIR, 'feature_importance.html')
fig.write_html(save_path_chart4)
print(f"Chart saved → {save_path_chart4}")
fig.show()

### CHART 5 — Precision-Recall Curves
Plots the Precision-Recall curves and calculates the Average Precision (AP) scores for all 6 models to evaluate performance under class imbalance.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

fig = go.Figure()

for model_name, filename in model_files.items():
    model_path = os.path.join(MODELS_DIR, filename)
    model = joblib.load(model_path)
    
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.decision_function(X_test)
        
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    
    fig.add_trace(go.Scatter(
        x=recall, y=precision,
        mode='lines',
        name=f"{model_name} (AP = {ap:.2f})"
    ))

fig.update_layout(
    title="Precision-Recall Curves — All Models",
    xaxis_title="Recall",
    yaxis_title="Precision",
    template="plotly_white",
    legend=dict(x=0.1, y=0.1)
)

# Save chart as HTML
save_path_chart5 = os.path.join(VISUALS_DIR, 'precision_recall.html')
fig.write_html(save_path_chart5)
print(f"Chart saved → {save_path_chart5}")
fig.show()

### FINAL CELL — Champion Model Selection

In [ ]:
from IPython.display import display, Markdown

# The results_df is sorted descending by F1 Score.
# Thus, index 0 contains the model with the highest F1 score.
champion_row = results_df.iloc[0]
champion_name = champion_row['Model']
champion_acc = champion_row['Accuracy']
champion_prec = champion_row['Precision']
champion_rec = champion_row['Recall']
champion_f1 = champion_row['F1']
champion_auc = champion_row['ROC-AUC']

conclusion_md = f"""
## Champion Model Selection

Based on the empirical results, the champion model is **{champion_name}**.

### Exact Metric Scores:
- **Accuracy**: {champion_acc:.4f}
- **Precision**: {champion_prec:.4f}
- **Recall**: {champion_rec:.4f}
- **F1 Score**: {champion_f1:.4f}
- **ROC-AUC**: {champion_auc:.4f}

### Business Justification:
For predicting customer churn, identifying as many at-risk customers as possible (high Recall) is the primary objective to prevent revenue loss.
However, we must balance this with Precision to avoid wasting retention budgets on stable customers.
**{champion_name}** provides the optimal trade-off by maximizing the F1 score, which ensures both high coverage of actual churners and cost-effective campaign spend.
"""

display(Markdown(conclusion_md))